In [7]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# 1. Cấu hình các tùy chọn cho Chrome để giống người dùng thật nhất
chrome_options = Options()
# chrome_options.add_argument("--headless") # Bỏ comment dòng này nếu bạn muốn chạy ẩn danh (không hiện cửa sổ Chrome)
chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Ẩn dấu vết bot của Selenium
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# 2. Khởi tạo WebDriver (Tự động tải Driver phù hợp với phiên bản Chrome máy bạn)
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

url = "https://thuvienphapluat.vn/chinh-sach-phap-luat-moi/vn/ho-tro-phap-luat/tu-van-phap-luat/42248/danh-muc-luat-bo-luat-hien-hanh-tai-viet-nam"

try:
    print("Đang mở trình duyệt và tải trang...")
    driver.get(url)
    
    # Chờ 5 giây để trang tải xong hoàn toàn và bypass Cloudflare (nếu có)
    print("Đang chờ trang tải và xử lý bảo mật...")
    time.sleep(5)
    
    # 3. Lấy toàn bộ HTML sau khi trình duyệt đã render xong
    html_source = driver.page_source
    
    # Lưu file HTML thô
    with open("thuvienphapluat_selenium.html", "w", encoding="utf-8") as file:
        file.write(html_source)
    print("-> Đã lưu file HTML thô thành công: thuvienphapluat_selenium.html")
    
    # 4. Trích xuất nội dung chính bằng BeautifulSoup
    soup = BeautifulSoup(html_source, 'html.parser')
    content_div = soup.find('div', class_='content-news') or soup.find('article') or soup.find('div', id='content')
    
    if content_div:
        with open("noidung_luat.html", "w", encoding="utf-8") as file:
            file.write(str(content_div))
        print("-> Đã lọc và lưu riêng HTML phần nội dung: noidung_luat.html")
    else:
        print("Không tìm thấy thẻ chứa nội dung chính bài viết, nhưng file raw đã được lưu.")

except Exception as e:
    print(f"Đã xảy ra lỗi trong quá trình cào dữ liệu: {e}")

finally:
    # Đóng trình duyệt sau khi hoàn thành
    driver.quit()
    print("Đã đóng trình duyệt.")

Đang mở trình duyệt và tải trang...
Đang chờ trang tải và xử lý bảo mật...
-> Đã lưu file HTML thô thành công: thuvienphapluat_selenium.html
Không tìm thấy thẻ chứa nội dung chính bài viết, nhưng file raw đã được lưu.
Đã đóng trình duyệt.


**EXTRACT LINKS FROM VERY FIRST LINK**

In [10]:
import json
from bs4 import BeautifulSoup

# Đường dẫn tới file HTML bạn đã tải về máy trước đó
input_html_file = "thuvienphapluat_selenium.html"
output_links_file = "danh_sach_luat.json"

def extract_links_from_file():
    print(f"Đang đọc dữ liệu từ file {input_html_file}...")
    
    with open(input_html_file, "r", encoding="utf-8") as f:
        html_content = f.read()
        
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Tìm vùng chứa bài viết (dựa theo cấu trúc class='newcontent' bạn cung cấp)
    content_div = soup.find('div', class_='newcontent')
    if not content_div:
        print("Không tìm thấy thẻ div.newcontent, chuyển sang tìm toàn bộ thẻ <a>")
        content_div = soup
        
    all_links = content_div.find_all('a')
    law_targets = []
    
    for link in all_links:
        href = link.get('href', '')
        text = link.text.strip()
        
        # Chỉ lọc các link dẫn tới văn bản pháp luật và tiêu đề chứa chữ "Luật" hoặc "Bộ luật"
        if "van-ban/" in href and ("Luật" in text or "Bộ luật" in text):
            # Chuẩn hóa link đầy đủ
            if not href.startswith("http"):
                href = "https://thuvienphapluat.vn" + href
                
            # Loại bỏ trùng lặp dữ liệu
            item = {"name": text, "url": href}
            if item not in law_targets:
                law_targets.append(item)
                
    # Lưu danh sách thành file JSON để bước 2 sử dụng
    with open(output_links_file, "w", encoding="utf-8") as f:
        json.dump(law_targets, f, ensure_ascii=False, indent=4)
        
    print(f"-> Hoàn thành! Đã tìm thấy {len(law_targets)} luật và lưu vào file '{output_links_file}'")

if __name__ == "__main__":
    extract_links_from_file()

Đang đọc dữ liệu từ file thuvienphapluat_selenium.html...
-> Hoàn thành! Đã tìm thấy 256 luật và lưu vào file 'danh_sach_luat.json'


**REQUEST EACH LINK**

In [11]:
import os
import json
import time
import random
import re
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

def clean_filename(filename):
    """Xóa ký tự đặc biệt để tạo tên file hợp lệ"""
    return re.sub(r'[\\/*?:"<>|]', "", filename).strip()

def main():
    links_file = "danh_sach_luat.json"
    output_folder = "output"
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    # Đọc danh sách link từ Bước 1
    if not os.path.exists(links_file):
        print(f"Không tìm thấy file {links_file}! Hãy chạy Bước 1 trước.")
        return
        
    with open(links_file, "r", encoding="utf-8") as f:
        law_targets = json.load(f)
        
    print(f"Tìm thấy {len(law_targets)} link chuẩn bị crawl...")
    
    # Khởi tạo trình duyệt chống phát hiện (Undetected Chromedriver)
    print("Đang khởi tạo trình duyệt ẩn mình...")
    options = uc.ChromeOptions()
    # options.add_argument('--headless') # Bỏ comment nếu muốn chạy ngầm hoàn toàn
    options.add_argument('--disable-gpu')
    options.add_argument('--start-maximized')
    
    driver = uc.Chrome(options=options)
    
    try:
        for index, law in enumerate(law_targets, start=1):
            law_name = law["name"]
            law_url = law["url"]
            
            print(f"\n[{index}/{len(law_targets)}] Đang truy cập: {law_name}")
            
            # 1. Đi tới link luật
            driver.get(law_url)
            
            # 2. Giả lập hành vi cuộn chuột (Human behavior)
            # Cuộn xuống một chút để kích hoạt load nội dung đầy đủ
            time.sleep(random.uniform(1.5, 3.0))
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 4);")
            time.sleep(random.uniform(1.0, 2.0))
            
            # 3. Lấy mã nguồn HTML đã render xong
            html_source = driver.page_source
            
            # 4. Lưu file thành từng file html riêng biệt
            safe_name = clean_filename(law_name)
            file_path = os.path.join(output_folder, f"{safe_name}.html")
            
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(html_source)
            print(f"   => Đã lưu thành công: {file_path}")
            
            # 5. Nghỉ ngẫu nhiên để đánh lừa AI chống bot
            sleep_time = random.uniform(4.0, 8.0)
            print(f"   => Nghỉ ngẫu nhiên {sleep_time:.2f} giây trước khi sang link tiếp theo...")
            time.sleep(sleep_time)
            
    except Exception as e:
        print(f"Có lỗi xảy ra: {e}")
        
    finally:
        driver.quit()
        print("\n Đã hoàn thành tải dữ liệu và đóng trình duyệt!")

if __name__ == "__main__":
    main()

Tìm thấy 256 link chuẩn bị crawl...
Đang khởi tạo trình duyệt ẩn mình...

[1/256] Đang truy cập: Luật ban hành văn bản quy phạm pháp luật 2015
   => Đã lưu thành công: output\Luật ban hành văn bản quy phạm pháp luật 2015.html
   => Nghỉ ngẫu nhiên 5.84 giây trước khi sang link tiếp theo...

[2/256] Đang truy cập: Luật Đất đai 2024
   => Đã lưu thành công: output\Luật Đất đai 2024.html
   => Nghỉ ngẫu nhiên 4.33 giây trước khi sang link tiếp theo...

[3/256] Đang truy cập: Luật Các tổ chức tín dụng 2024
   => Đã lưu thành công: output\Luật Các tổ chức tín dụng 2024.html
   => Nghỉ ngẫu nhiên 4.86 giây trước khi sang link tiếp theo...

[4/256] Đang truy cập: Luật sửa đổi Luật Đất đai, Luật Nhà ở, Luật Kinh doanh bất động sản và Luật Các tổ chức tín dụng 2024
   => Đã lưu thành công: output\Luật sửa đổi Luật Đất đai, Luật Nhà ở, Luật Kinh doanh bất động sản và Luật Các tổ chức tín dụng 2024.html
   => Nghỉ ngẫu nhiên 4.29 giây trước khi sang link tiếp theo...

[5/256] Đang truy cập: Luật 

In [12]:
import os
import json
import time
import random
import re
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

def clean_filename(filename):
    """Xóa ký tự đặc biệt để tạo tên file hợp lệ"""
    return re.sub(r'[\\/*?:"<>|]', "", filename).strip()

def main():
    links_file = "danh_sach_luat.json"
    output_folder = "output"
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    # Đọc danh sách link từ Bước 1
    if not os.path.exists(links_file):
        print(f"Không tìm thấy file {links_file}! Hãy chạy Bước 1 trước.")
        return
        
    with open(links_file, "r", encoding="utf-8") as f:
        law_targets = json.load(f)
        
    print(f"Tổng số luật trong danh sách: {len(law_targets)} link.")
    
    # Khởi tạo trình duyệt chống phát hiện (Undetected Chromedriver)
    print("Đang khởi tạo trình duyệt ẩn mình...")
    options = uc.ChromeOptions()
    # options.add_argument('--headless') # Bỏ comment nếu muốn chạy ẩn hoàn toàn
    options.add_argument('--disable-gpu')
    options.add_argument('--start-maximized')
    
    driver = uc.Chrome(options=options)
    
    # CẤU HÌNH QUAN TRỌNG: Giới hạn thời gian chờ tải trang là 35 giây để tránh bị treo vô hạn
    driver.set_page_load_timeout(35)
    
    success_count = 0 # Đếm số file tải thành công trong phiên này để cho nghỉ dài
    
    try:
        for index, law in enumerate(law_targets, start=1):
            law_name = law["name"]
            law_url = law["url"]
            
            # Tạo tên file trước để kiểm tra xem đã crawl chưa
            safe_name = clean_filename(law_name)
            file_path = os.path.join(output_folder, f"{safe_name}.html")
            
            # TÍNH NĂNG 1: Kiểm tra nếu file đã tồn tại thì bỏ qua (Crawl tiếp sức)
            if os.path.exists(file_path) and os.path.getsize(file_path) > 1000:
                print(f"[{index}/{len(law_targets)}] Đã có sẵn -> Bỏ qua: {law_name}")
                continue
                
            print(f"\n[{index}/{len(law_targets)}] Đang truy cập: {law_name}")
            
            try:
                # 1. Đi tới link luật
                driver.get(law_url)
                
                # 2. Giả lập hành vi cuộn chuột (Human behavior)
                time.sleep(random.uniform(2.0, 3.5))
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 5);")
                time.sleep(random.uniform(1.5, 2.5))
                
                # 3. Lấy mã nguồn HTML đã render xong
                html_source = driver.page_source
                
                # 4. Lưu file html
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(html_source)
                print(f"   => Đã lưu thành công: {file_path}")
                
                success_count += 1
                
                # TÍNH NĂNG 3: Cứ tải được 10 file mới thì cho script nghỉ dài 60 giây để reset hệ thống
                if success_count % 10 == 0:
                    print("\n[AN TOÀN] Đã tải liên tục 10 file. Script tạm nghỉ 60 giây để tránh bị hệ thống quét...")
                    time.sleep(60)
                    continue
                
                # 5. Nghỉ ngẫu nhiên giữa các link thông thường
                sleep_time = random.uniform(5.0, 9.0)
                print(f"   => Nghỉ ngẫu nhiên {sleep_time:.2f} giây trước khi sang link tiếp theo...")
                time.sleep(sleep_time)
                
            except Exception as page_error:
                # Nếu trang này bị timeout hoặc lỗi, in thông báo lỗi và nhảy sang trang tiếp theo chứ không sập cả chương trình
                print(f"   => [LỖI BỎ QUA] Không thể tải luật '{law_name}' do: {page_error}")
                print("   => Tự động chuyển sang link kế tiếp sau 5 giây...")
                time.sleep(5)
                continue
            
    except Exception as e:
        print(f"Có lỗi hệ thống nghiêm trọng xảy ra: {e}")
        
    finally:
        driver.quit()
        print("\n Đã đóng trình duyệt và hoàn thành tiến trình!")

if __name__ == "__main__":
    main()

Tổng số luật trong danh sách: 256 link.
Đang khởi tạo trình duyệt ẩn mình...
[1/256] Đã có sẵn -> Bỏ qua: Luật ban hành văn bản quy phạm pháp luật 2015
[2/256] Đã có sẵn -> Bỏ qua: Luật Đất đai 2024
[3/256] Đã có sẵn -> Bỏ qua: Luật Các tổ chức tín dụng 2024
[4/256] Đã có sẵn -> Bỏ qua: Luật sửa đổi Luật Đất đai, Luật Nhà ở, Luật Kinh doanh bất động sản và Luật Các tổ chức tín dụng 2024
[5/256] Đã có sẵn -> Bỏ qua: Luật Lực lượng tham gia bảo vệ an ninh, trật tự ở cơ sở 2023
[6/256] Đã có sẵn -> Bỏ qua: Luật Kinh doanh bất động sản 2023
[7/256] Đã có sẵn -> Bỏ qua: Luật Tài nguyên nước 2023
[8/256] Đã có sẵn -> Bỏ qua: Luật Căn cước 2023
[9/256] Đã có sẵn -> Bỏ qua: Luật Nhà ở 2023
[10/256] Đã có sẵn -> Bỏ qua: Luật Viễn thông 2023
[11/256] Đã có sẵn -> Bỏ qua: Luật Giao dịch điện tử 2023
[12/256] Đã có sẵn -> Bỏ qua: Luật Phòng thủ dân sự 2023
[13/256] Đã có sẵn -> Bỏ qua: Luật Bảo vệ quyền lợi người tiêu dùng 2023

[14/256] Đang truy cập: Luật Hợp tác xã 2023
   => Đã lưu thành công:

**JSONIFY**


In [ ]:
import json
import re
from bs4 import BeautifulSoup

def structurize_law_v2(input_file, output_file):
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            html_content = f.read()
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file {input_file}")
        return

    soup = BeautifulSoup(html_content, 'html.parser')
    
    # 1. Nhắm chính xác vào vùng chứa nội dung văn bản luật
    content_area = soup.find('div', {'data-role': 'content-body'})
    if not content_area:
        # Fallback nếu không tìm thấy div chuyên dụng
        content_area = soup.find('article', class_='the-document')

    articles = []
    current_article = None

    # 2. Duyệt qua các thẻ p vì mỗi dòng nội dung thường nằm trong 1 thẻ p
    paragraphs = content_area.find_all('p')

    for p in paragraphs:
        text = p.get_text(strip=True)
        if not text:
            continue

        # --- NHẬN DIỆN ĐIỀU ---
        # Tìm trong thẻ p có span class 'demuc4' hoặc text bắt đầu bằng "Điều X."
        is_demuc4 = p.find('span', class_='demuc4')
        article_match = re.match(r'^Điều\s+(\d+)[\.\s]+(.*)', text, re.IGNORECASE)
        
        if is_demuc4 or article_match:
            match = article_match if article_match else re.match(r'^Điều\s+(\d+)[\.\s]+(.*)', text, re.IGNORECASE)
            if match:
                article_num = int(match.group(1))
                article_title = match.group(2).strip()
                
                current_article = {
                    "article": article_num,
                    "title": article_title,
                    "clauses": []
                }
                articles.append(current_article)
                continue

        # --- NHẬN DIỆN KHOẢN ---
        if current_article is not None:
            # Tìm các dòng bắt đầu bằng "1. ", "2. " (Khoản)
            clause_match = re.match(r'^(\d+)\.\s+(.*)', text)
            
            if clause_match:
                clause_num = int(clause_match.group(1))
                clause_text = clause_match.group(2).strip()
                
                clause_obj = {
                    "clause": clause_num,
                    "id": f"Dieu_{current_article['article']}_Khoan_{clause_num}",
                    "text": clause_text
                }
                current_article["clauses"].append(clause_obj)
            else:
                # Xử lý nội dung bổ sung (điểm a, b, c...) hoặc văn bản không có số khoản
                if not current_article["clauses"]:
                    # Nếu Điều vừa tạo mà dòng tiếp theo không có số 1., coi đó là Khoản 1
                    current_article["clauses"].append({
                        "clause": 1,
                        "id": f"Dieu_{current_article['article']}_Khoan_1",
                        "text": text
                    })
                else:
                    # Cộng dồn nội dung vào khoản hiện tại (bao gồm các điểm a, b, c)
                    current_article["clauses"][-1]["text"] += f" {text}"

    # 3. Xuất file JSON
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)

    print(f"--- THÀNH CÔNG ---")
    print(f"Đã trích xuất {len(articles)} điều. File: {output_file}")

# --- CHẠY CODE ---
# Lưu ý: Tạo file 'law_source.txt' và dán nội dung HTML của bạn vào đó trước khi chạy
structurize_law_v2('law_source.txt', 'luat_lao_dong.json')

--- THÀNH CÔNG ---
Đã trích xuất 220 điều. File: luat_lao_dong.json
